# V3 sweep analysis

This notebook loads the v3 sweep output, filters the 1000 and 5000 iteration snapshots, validates the dataset, and writes the final CSV used for follow-up analysis.

## 1. Import Required Libraries
Import the libraries needed for data manipulation and CSV handling. This notebook is robust even if pandas is not installed in the local runtime.

In [ ]:
try:
    import pandas as pd
except ImportError:  # pragma: no cover
    pd = None
import csv
from pathlib import Path

src_path = Path('tmp/v3_sweep.csv')
final_path = Path('tmp/v3_sweep_final.csv')
print(f'source: {src_path}')
print(f'final: {final_path}')

source: tmp/v3_sweep.csv
final: tmp/v3_sweep_final.csv


## 2. Define Input Data or Parameters
Set the sweep file, output file, and the iteration checkpoints to keep in the final analysis.

In [ ]:
source_csv = Path('tmp/v3_sweep.csv')
final_csv = Path('tmp/v3_sweep_final.csv')
iters_to_keep = [1000, 5000]
required_columns = ['label', 'context', 'board', 'iteration', 'fold', 'check_call', 'bet_raise']
print(source_csv)
print(final_csv)
print(iters_to_keep)

tmp/v3_sweep.csv
tmp/v3_sweep_final.csv
[1000, 5000]


## 3. Generate 1000-Row Dataset
Create the filtered 1000-iteration snapshot from the larger sweep output.

In [ ]:
with source_csv.open(newline='') as f:
    rows = list(csv.DictReader(f))

df_1000 = [row for row in rows if int(row['iteration']) == 1000]
assert len(df_1000) == 9, f'Expected 9 rows at 1000 iters, found {len(df_1000)}'
print(f'1000-iter rows: {len(df_1000)}')
print(df_1000[:3])

1000-iter rows: 9
[{'label': 'dry_broadway', 'context': 'flop_open', 'board': 'Ah|Kd|2c', 'iteration': '1000', 'fold': '0.0', 'check_call': '0.40131495901050673', 'bet_raise': '0.5986850409894933'}, {'label': 'dry_broadway', 'context': 'flop_check', 'board': 'Ah|Kd|2c', 'iteration': '1000', 'fold': '0.0', 'check_call': '0.40131495901050673', 'bet_raise': '0.5986850409894933'}, {'label': 'dry_broadway', 'context': 'response_to_bet', 'board': 'Ah|Kd|2c', 'iteration': '1000', 'fold': '0.08773394150241703', 'check_call': '0.40372286885402425', 'bet_raise': '0.5085431896435587'}]


## 4. Generate 5000-Row Dataset
Create the filtered 5000-iteration snapshot from the same source file.

In [ ]:
with source_csv.open(newline='') as f:
    rows = list(csv.DictReader(f))

df_5000 = [row for row in rows if int(row['iteration']) == 5000]
assert len(df_5000) == 9, f'Expected 9 rows at 5000 iters, found {len(df_5000)}'
print(f'5000-iter rows: {len(df_5000)}')
print(df_5000[:3])

5000-iter rows: 9
[{'label': 'dry_broadway', 'context': 'flop_open', 'board': 'Ah|Kd|2c', 'iteration': '5000', 'fold': '0.0', 'check_call': '0.40131439805799624', 'bet_raise': '0.5986856019420038'}, {'label': 'dry_broadway', 'context': 'flop_check', 'board': 'Ah|Kd|2c', 'iteration': '5000', 'fold': '0.0', 'check_call': '0.40131439805799624', 'bet_raise': '0.5986856019420038'}, {'label': 'dry_broadway', 'context': 'response_to_bet', 'board': 'Ah|Kd|2c', 'iteration': '5000', 'fold': '0.08773254544893737', 'check_call': '0.40372326896721167', 'bet_raise': '0.508544185583851'}]


## 5. Combine and Validate Data
Concatenate the two filtered datasets and validate rows, columns, missing values, and duplicates before export.

In [ ]:
combined_rows = df_1000 + df_5000
assert len(combined_rows) == 18, f'Expected 18 combined rows, found {len(combined_rows)}'
assert all(set(required_columns).issubset(set(row.keys())) for row in combined_rows)
assert all(row.get('iteration') in {'1000', '5000'} for row in combined_rows)
assert all(row.get(k) is not None for row in combined_rows for k in required_columns)
print(f'Combined rows: {len(combined_rows)}')
print(combined_rows[:2])

Combined rows: 18
[{'label': 'dry_broadway', 'context': 'flop_open', 'board': 'Ah|Kd|2c', 'iteration': '1000', 'fold': '0.0', 'check_call': '0.40131495901050673', 'bet_raise': '0.5986850409894933'}, {'label': 'dry_broadway', 'context': 'flop_check', 'board': 'Ah|Kd|2c', 'iteration': '1000', 'fold': '0.0', 'check_call': '0.40131495901050673', 'bet_raise': '0.5986850409894933'}]


## 6. Clean and Standardize Columns
Rename the action columns into final reporting names and normalize the board string formatting.

In [ ]:
cleaned_rows = []
for row in combined_rows:
    cleaned = dict(row)
    cleaned['fold_prob'] = float(cleaned.pop('fold'))
    cleaned['check_call_prob'] = float(cleaned.pop('check_call'))
    cleaned['bet_raise_prob'] = float(cleaned.pop('bet_raise'))
    cleaned['board'] = cleaned['board'].replace('|', ' ')
    cleaned['iteration'] = int(cleaned['iteration'])
    cleaned_rows.append(cleaned)

print(cleaned_rows[:2])

[{'label': 'dry_broadway', 'context': 'flop_open', 'board': 'Ah Kd 2c', 'iteration': 1000, 'fold_prob': 0.0, 'check_call_prob': 0.40131495901050673, 'bet_raise_prob': 0.5986850409894933}, {'label': 'dry_broadway', 'context': 'flop_check', 'board': 'Ah Kd 2c', 'iteration': 1000, 'fold_prob': 0.0, 'check_call_prob': 0.40131495901050673, 'bet_raise_prob': 0.5986850409894933}]


## 7. Export Final CSV
Write the final combined dataset to a dedicated output CSV in the tmp directory.

In [ ]:
final_csv.parent.mkdir(parents=True, exist_ok=True)
fieldnames = ['label', 'context', 'board', 'iteration', 'fold_prob', 'check_call_prob', 'bet_raise_prob']
with final_csv.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in cleaned_rows:
        writer.writerow({
            'label': row['label'],
            'context': row['context'],
            'board': row['board'],
            'iteration': row['iteration'],
            'fold_prob': row['fold_prob'],
            'check_call_prob': row['check_call_prob'],
            'bet_raise_prob': row['bet_raise_prob'],
        })
print(f'Wrote: {final_csv}')
print(f'Rows written: {len(cleaned_rows)}')

Wrote: tmp/v3_sweep_final.csv
Rows written: 18


# Plot policy trajectories
This section plots all action-family curves across the full sweep, and then isolates the response-to-bet context for a cleaner read.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

full_df = pd.read_csv('tmp/v3_sweep.csv')
full_df['board_label'] = full_df['label'] + ' / ' + full_df['context']

for metric in ['fold', 'check_call', 'bet_raise']:
    fig, ax = plt.subplots(figsize=(10, 5))
    for (label, context), group in full_df.groupby(['label', 'context']):
        by_iter = group.sort_values('iteration')
        ax.plot(by_iter['iteration'], by_iter[metric], label=f'{label} / {context}', marker='o', linewidth=1.5)
    ax.set_title(f'V3 sweep: {metric}')
    ax.set_xlabel('iteration')
    ax.set_ylabel(metric)
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.show()

/var/folders/5m/q6fqtkss6kqfktc2d8718hr80000gn/T/ipykernel_31482/983410784.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/5m/q6fqtkss6kqfktc2d8718hr80000gn/T/ipykernel_31482/983410784.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/5m/q6fqtkss6kqfktc2d8718hr80000gn/T/ipykernel_31482/983410784.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
response_df = full_df[full_df['context'] == 'response_to_bet'].copy()
response_df = response_df.sort_values(['label', 'iteration'])

for metric in ['fold', 'check_call', 'bet_raise']:
    fig, ax = plt.subplots(figsize=(9, 5))
    for label, group in response_df.groupby('label'):
        by_iter = group.sort_values('iteration')
        ax.plot(by_iter['iteration'], by_iter[metric], label=label, marker='o', linewidth=2)
    ax.set_title(f'Response-to-bet context: {metric}')
    ax.set_xlabel('iteration')
    ax.set_ylabel(metric)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()

/var/folders/5m/q6fqtkss6kqfktc2d8718hr80000gn/T/ipykernel_31482/1741411844.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/5m/q6fqtkss6kqfktc2d8718hr80000gn/T/ipykernel_31482/1741411844.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/5m/q6fqtkss6kqfktc2d8718hr80000gn/T/ipykernel_31482/1741411844.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
summary = (
    full_df.groupby(['label', 'context', 'iteration'])
    .agg({'fold': 'mean', 'check_call': 'mean', 'bet_raise': 'mean'})
    .reset_index()
)
final_rows = summary[summary['iteration'].isin([1000, 5000])].copy()
final_rows = final_rows.sort_values(['label', 'context', 'iteration']).reset_index(drop=True)
final_rows[['label', 'context', 'iteration', 'fold', 'check_call', 'bet_raise']]

,label,context,iteration,fold,check_call,bet_raise
0,dry_broadway,flop_check,1000,0.000000,0.401315,0.598685
1,dry_broadway,flop_check,5000,0.000000,0.401314,0.598686
2,dry_broadway,flop_open,1000,0.000000,0.401315,0.598685
3,dry_broadway,flop_open,5000,0.000000,0.401314,0.598686
4,dry_broadway,response_to_bet,1000,0.087734,0.403723,0.508543
5,dry_broadway,response_to_bet,5000,0.087733,0.403723,0.508544
6,paired_board,flop_check,1000,0.000000,0.401271,0.598729
7,paired_board,flop_check,5000,0.000000,0.401270,0.598730
8,paired_board,flop_open,1000,0.000000,0.401271,0.598729
9,paired_board,flop_open,5000,0.000000,0.401270,0.598730
